In [ ]:
# ============================================================================
# PARAMETERS — this cell is identical in all three notebooks.
# ============================================================================
NOTEBOOK_NAME = "01_setup_and_data"    # identity + build fingerprint of THESE cells;
NOTEBOOK_BUILD = "1ac18282a431"  # checked against the repo so stale cells fail loudly
RUN_MODE = "micro"          # "smoke" | "micro" (default) | "budget" | "full"
NUM_GPUS = None             # None = use every visible GPU; set 1 to force single-GPU
PUBLISH_KAGGLE_DATASET = True
CKPT_DATASET_SLUG = "dentex-repro-ckpts"
DATA_DATASET_SLUG = "dentex-repro-data"
REPO_URL = "https://github.com/christopherh-88/HierarchicalDet.git"

import os, subprocess, sys

# On Kaggle the repo is cloned into /kaggle/working (the only writable place
# that survives "Save Version"); locally the notebook already sits inside it.
if os.path.isdir("/kaggle/working"):
    CLONE = "/kaggle/working/repo"
    if os.path.isdir(os.path.join(CLONE, ".git")):
        subprocess.run(["git", "-C", CLONE, "pull", "--ff-only"], check=False)
    else:
        subprocess.run(["git", "clone", "--depth", "50", REPO_URL, CLONE], check=True)
    PROJECT_ROOT = os.path.join(CLONE, "dentex-repro")
else:
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
os.environ["RUN_MODE"] = RUN_MODE
print("project root:", PROJECT_ROOT)


In [ ]:
# ---- Environment: install, pin, and prove the VENDORED code is what loaded ----
# Kaggle reverts to its base image every session, so this runs every time.
import json
from src import setup_env

# `git pull` above refreshed src/ and configs_repro/ -- but NOT these cells,
# which are the copy uploaded to Kaggle. Fail loudly rather than run a mix.
print("notebook build:", setup_env.assert_notebook_current(NOTEBOOK_NAME, NOTEBOOK_BUILD))
setup_env.install_dependencies()
# The vendored pycocotools ships Python sources only; its compiled `_mask`
# extension is grafted in here and VERIFIED BY IMPORT. It is compiled against
# numpy's C ABI, so a mismatch surfaces as "numpy.dtype size changed" deep
# inside detectron2.structures — which reads as a detectron2 problem and is not.
import numpy
print("numpy {} | pycocotools _mask -> {}".format(
    numpy.__version__, setup_env.ensure_pycocotools_mask()))
run = setup_env.bootstrap(RUN_MODE)

from src import manifest, train_utils

NUM_GPUS = NUM_GPUS if NUM_GPUS is not None else max(1, train_utils.visible_gpus())
lock = setup_env.write_requirements_lock()
environment = setup_env.env_report()
manifest.record_environment(environment)

# The repo vendors MODIFIED detectron2 / pycocotools (multi-label partial
# annotations, 3-tier category schema). A pip-installed copy silently shadows
# them and every number changes, so this is an assertion, not a warning.
found = setup_env.assert_vendored()
for module, path in found.items():
    print("{:14s} -> {}".format(module, path))
import detectron2, pycocotools, evaluator                              # noqa: F401
from hierarchialdet.util.coco_3class_eval import COCOEvaluator         # noqa: F401
from hierarchialdet.dataset_mapper_patched import DiffusionDetDatasetMapper  # noqa: F401
print("full import chain OK | commit {} | {} GPU process(es)".format(
    environment["repo_commit"][:12], NUM_GPUS))


In [ ]:
# ---- Download and extract ----
# Marker-gated at every step, so a session kill resumes rather than restarts.
from src import data_convert

RAW = os.path.join(setup_env.PROJECT_ROOT, "dentex_raw")
download = data_convert.download_and_extract(
    RAW, include_unlabelled=False, delete_archives=True)
print(json.dumps({k: v for k, v in download.items() if k != "raw_train_json"}, indent=2))


In [ ]:
# ---- Convert every split into the normalized 3-tier schema ----
# quadrant            flat categories/category_id      -> 3-tier (remapped BY NAME:
#                                                         a raw id copy would swap
#                                                         quadrants 1 and 2)
# quadrant_enumeration  categories_1/2                 -> 3-tier
# diagnosis             already 3-tier                 -> copied
# test split            LabelMe polygons, Turkish text -> 3-tier
conversion = data_convert.convert_all(download["raw_train_json"], download["test_label_dir"])
paths = conversion["paths"]
report = conversion["test_parse_report"]
print("test split parsed: {} images, {} annotations kept, {} distinct raw labels"
      .format(report["images"], report["annotations"], report["distinct_raw_labels"]))

# The raw test labels use a 9-code clinical scheme; only codes 1/6/7 are task
# classes. Everything else is excluded BY NAME and counted — never silently.
print("\nexcluded out-of-task annotations ({} total):".format(report["excluded_total"]))
for word, info in report["excluded_out_of_task"].items():
    print("  {:10s} {:4d}  ({})".format(word, info["count"], info["meaning"]))
print("class-code inconsistencies:", report["class_code_inconsistencies"])

print("\ntest-split diagnosis ground truth:")
for name, count in report["diagnosis_histogram"].items():
    print("  {:20s} {:4d}".format(name, count))

missing = report["diagnosis_classes_without_ground_truth"]
if missing:
    print("\n*** FINDING: no test ground truth for {} ***".format(missing))
    print(data_convert.TEST_LABEL_FINDING)
    setup_env.log_deviation(
        "test-split ground truth missing entirely for {}".format(missing),
        data_convert.TEST_LABEL_FINDING,
        "01_setup_and_data",
        impact="test-split evaluation covers {} of 4 diagnosis classes; the "
               "missing class contributes no ground truth, so its per-class AP "
               "is undefined and the diagnosis-tier mean AP averages over the "
               "classes that have ground truth".format(4 - len(missing)))

for raw, parsed in list(report["parsed_examples"].items())[:12]:
    print("  {!r:26s} -> q{} t{} {}".format(raw, *parsed))


In [ ]:
# ---- Audit each split ----
audits = {}
for name, (json_key, image_key) in [
    ("quadrant_train", ("train_quadrant", "img_quadrant")),
    ("quadrant_enumeration_train", ("train_enumeration", "img_enumeration")),
    ("diagnosis_train", ("train_diagnosis", "img_diagnosis")),
    ("diagnosis_val", ("val_diagnosis", "img_val")),
    ("diagnosis_test", ("test_diagnosis", "img_test")),
]:
    audits[name] = data_convert.audit_split(paths[json_key], paths[image_key], name)
    audit = audits[name]
    print("{:28s} {:>5} images {:>6} annotations  tier coverage {}  zero-ann {}".format(
        name, audit["num_images"], audit["num_annotations"],
        audit["tier_coverage"], audit["images_with_zero_annotations"]))
    if audit["unreadable_images"] or audit["malformed_boxes"] or audit["missing_image_files"]:
        print("    PROBLEMS: unreadable={} malformed_boxes={} missing_files={}".format(
            len(audit["unreadable_images"]), len(audit["malformed_boxes"]),
            len(audit["missing_image_files"])))


In [ ]:
# ---- Hard assertions: composition, and the presence of test ground truth ----
actual_counts = data_convert.assert_published_counts(
    audits, download.get("unlabelled_in_zip"))
print("published composition confirmed:", json.dumps(actual_counts, indent=2))

data_convert.assert_test_ground_truth(audits["diagnosis_test"])
print("test ground truth present: {} annotations, {} with a diagnosis label".format(
    audits["diagnosis_test"]["num_annotations"],
    audits["diagnosis_test"]["tier_coverage"]["tier2_diagnosis"]))

print()
print(data_convert.LICENSE_NOTE)
setup_env.log_deviation(
    "DENTEX license discrepancy (CC BY-SA on GitHub vs CC BY-NC-SA on HuggingFace)",
    "the two published statements disagree; the stricter non-commercial reading is assumed",
    "01_setup_and_data")


In [ ]:
# ---- Registration check: identical class indices across every split ----
# The model has fixed-size heads shared across curriculum stages, so drifting
# class indices between splits would let weight transfer silently remap labels
# with no error raised.
from src import registration

registration_report = registration.verify_registration()
print(json.dumps(registration_report["thing_classes"], indent=2))


In [ ]:
# ---- Clean and stress evaluation subsets (rule written down, ids saved) ----
subsets = data_convert.build_clean_stress_subsets(paths["test_diagnosis"])
os.makedirs(paths["subsets"], exist_ok=True)
subset_path = os.path.join(paths["subsets"], "clean_stress.json")
with open(subset_path, "w") as handle:
    json.dump(subsets, handle, indent=2)

for which in ("clean", "stress"):
    out = os.path.join(paths["subsets"], "test_{}.json".format(which))
    data_convert.subset_json(paths["test_diagnosis"], subsets[which]["image_ids"], out)
    print("{:6s} {:>3} images -> {}".format(which, len(subsets[which]["image_ids"]), out))
print()
print(subsets["rule"])

# ---- Is "stress" different from "clean" on anything besides box count? ----
# The split is defined BY annotation count, which sits inside the AP
# denominator -- more ground truth per image gives more chances to match,
# so a "stress" AP above "clean" AP can be that mechanical effect, not the
# model handling stress cases better. Checked here, CPU-only, against the
# one confound (image resolution) that doesn't need a model.
confound = data_convert.clean_stress_confound_check(subsets, paths["test_diagnosis"])
print(json.dumps(confound, indent=2))
tables.write_table(
    "clean_stress_confound", tables.clean_stress_confound_rows(confound),
    ["subset", "n_images", "mean_annotations_per_image", "mean_image_area_px",
     "min_image_area_px", "max_image_area_px"],
    "Clean vs. stress subset comparison on axes outside the split's own "
    "definition. mean_annotations_per_image differs by construction (that is "
    "the split rule); mean_image_area_px close between the two rows means "
    "resolution is not a confound, but the split is still defined by box "
    "count, which sits inside the AP denominator -- a stress-subset AP above "
    "the clean-subset AP is not on its own evidence the model handles stress "
    "cases better.",
    "01_setup_and_data", run.mode, "table:clean_stress_confound",
    inputs=[paths["test_diagnosis"]])


In [ ]:
# ---- Ground-truth sanity figure: 5 random images per tier, boxes drawn ----
import random
from src import figures

random.seed(setup_env.BASE_SEED)
panels = []
for name, (json_key, image_key, tier) in {
    "quadrant": ("train_quadrant", "img_quadrant", 0),
    "enumeration": ("train_enumeration", "img_enumeration", 1),
    "diagnosis": ("test_diagnosis", "img_test", 2),
}.items():
    with open(paths[json_key]) as handle:
        data = json.load(handle)
    by_image = {}
    for annotation in data["annotations"]:
        by_image.setdefault(annotation["image_id"], []).append(annotation)
    names = {level: {c["id"]: str(c["name"])
                     for c in data["categories_{}".format(level + 1)]} for level in range(3)}
    chosen = random.sample([i for i in data["images"] if by_image.get(i["id"])], 5)
    for image in chosen:
        boxes = []
        for annotation in by_image[image["id"]]:
            label = "/".join(
                names[level].get(annotation.get("category_id_{}".format(level + 1)), "")
                for level in range(tier + 1)
                if annotation.get("category_id_{}".format(level + 1)) is not None)
            boxes.append((annotation["bbox"], label))
        panels.append({"image_path": os.path.join(paths[image_key], image["file_name"]),
                       "title": "{}: {}".format(name, image["file_name"]),
                       "gt": boxes, "pred": []})

figure = figures.overlay_grid(panels, columns=5, panel_height=1.7,
                              title="Ground-truth sanity check (5 images per tier)")
figures.save_figure(figure, "gt_sanity_check",
                    "Ground-truth annotations overlaid on five randomly sampled images "
                    "from each DENTEX tier. Labels read quadrant/enumeration/diagnosis.",
                    "01_setup_and_data", run.mode, "figure:gt_sanity",
                    inputs=[paths["train_quadrant"], paths["train_enumeration"],
                            paths["test_diagnosis"]])


In [ ]:
# ---- Dataset audit table, hashes, and publication ----
from src import tables

tables.write_table(
    "dataset_audit", tables.audit_rows(audits),
    ["split", "images", "annotations", "images_without_annotations",
     "quadrant_labels", "enumeration_labels", "diagnosis_labels",
     "distinct_resolutions", "unreadable_images", "missing_image_files",
     "malformed_boxes"],
    "DENTEX composition and integrity, as received and converted by this study.",
    "01_setup_and_data", run.mode, "table:dataset_audit", inputs=[paths["coco"]])

# Static -- no dataset or GPU needed, derived from the converter's own word
# tables -- so it is produced unconditionally, every run mode, every session.
tables.write_table(
    "label_scheme", tables.label_scheme_rows(),
    ["code", "turkish_word", "gloss", "task_class", "note"],
    "The DENTEX test split's 9-code Turkish clinical labelling scheme. "
    "task_class marks the 3 codes (1/6/7) the diagnosis task actually uses; "
    "Deep Caries has no code in this scheme at all.",
    "01_setup_and_data", run.mode, "table:label_scheme")

# ---- What is a tier-0 "quadrant box", precisely? ----
# A literal quarter-image region averages ~25% area; this answers whether
# that is what tier-0 actually evaluates, with numbers instead of a guess.
quadrant_geometry = data_convert.quadrant_box_geometry(paths["train_quadrant"])
print(json.dumps(quadrant_geometry, indent=2))
tables.write_table(
    "quadrant_box_geometry", tables.quadrant_geometry_rows(quadrant_geometry),
    ["quadrant_category_id", "n_boxes", "mean_area_fraction_of_image",
     "mean_center_x", "mean_center_y"],
    "Tier-0 quadrant box geometry (train split). mean_area_fraction_of_image "
    "near 0.25 would mean a literal quarter-image region; well below that "
    "means a tight bounding box around the visible teeth in that quadrant.",
    "01_setup_and_data", run.mode, "table:quadrant_box_geometry",
    inputs=[paths["train_quadrant"]])

# ---- Corruption pipeline visual check (CPU-only, no GPU or model needed) ----
# Answers "is this a pipeline bug" for the degradation grid by looking at what
# it actually produces, on real images, before any inference is spent on it.
from PIL import Image
from src import degradations

_check_names = sorted(f for f in os.listdir(paths["img_test"])
                      if f.lower().endswith((".png", ".jpg", ".jpeg")))[:2]
_check_conditions = [("none", None), ("blur", 4.0), ("downscale", 0.25), ("jpeg", 20)]
_check_panels = []
for _name in _check_names:
    with Image.open(os.path.join(paths["img_test"], _name)) as _image:
        _image.load()
        for _kind, _severity in _check_conditions:
            _shown = _image if _kind == "none" else degradations.degrade_image(_image, _kind, _severity)
            _label = "clean" if _kind == "none" else degradations.condition_label(_kind, _severity)
            _tmp_path = os.path.join(paths["audit"], "_corruption_check_{}_{}.png".format(_name, _label))
            _shown.save(_tmp_path, format="PNG")
            _check_panels.append({"image_path": _tmp_path,
                                  "title": "{}: {}".format(_name, _label), "gt": [], "pred": []})

_check_figure = figures.overlay_grid(_check_panels, columns=len(_check_conditions), panel_height=2.2,
                                     title="Corruption pipeline visual check (clean vs. degraded, no model)")
figures.save_figure(
    _check_figure, "corruption_visual_check",
    "Same two test images under each corruption condition, produced by the same "
    "code path notebook 03's degradation grid uses, with no model involved. "
    "Exists to rule out a pipeline defect (wrong mode, size, or channel "
    "corruption) as the explanation for degraded conditions occasionally "
    "outscoring clean in table:degradation, before spending any inference on it.",
    "01_setup_and_data", run.mode, "figure:corruption_visual_check")
for _name in _check_names:
    for _kind, _severity in _check_conditions:
        _label = "clean" if _kind == "none" else degradations.condition_label(_kind, _severity)
        os.remove(os.path.join(paths["audit"], "_corruption_check_{}_{}.png".format(_name, _label)))

# Raw per-class diagnosis counts, independent of any training run -- this is
# the table a reader checks to verify the Deep Caries finding themselves
# instead of taking the deviation log's word for it.
_histogram_rows = tables.diagnosis_label_histogram_rows(audits)
_histogram_columns = ["split"] + sorted({
    key for row in _histogram_rows for key in row if key != "split"
})
tables.write_table(
    "diagnosis_label_histogram", _histogram_rows, _histogram_columns,
    "Diagnosis-tier class counts per split, straight from the converted "
    "annotations. A class missing from a split's columns had zero occurrences "
    "in that split.",
    "01_setup_and_data", run.mode, "table:diagnosis_label_histogram",
    inputs=[paths["coco"]])

hashes = data_convert.dataset_hashes()
with open(os.path.join(paths["root"], "dataset_hashes.json"), "w") as handle:
    json.dump(hashes, handle, indent=2)
for name, digest in hashes.items():
    print("{:34s} {}".format(name, digest[:16]))


In [ ]:
# ---- Cross-tier / cross-split image duplication (CPU-only, no GPU) ----
# Tiers 0/1 (quadrant, quadrant-enumeration) are pooled entirely into
# training with no held-out split of their own (DENTEX paper). If the
# diagnosis-tier test/val images overlap those training tiers by content,
# the hierarchical pipeline may have already seen those pixels, under a
# different annotation tier, before "testing" on them -- a property of the
# original design and public release, not of this conversion.
overlap = data_convert.image_hash_overlap()
print(json.dumps(overlap, indent=2))
if overlap["total_image_files"]:
    test_pct = 100 * overlap["test_overlap_with_training"] / max(
        1, overlap["per_split_unique_files"]["diagnosis_test"])
    val_pct = 100 * overlap["val_overlap_with_training"] / max(
        1, overlap["per_split_unique_files"]["diagnosis_val"])
    if overlap["test_overlap_with_training"] or overlap["val_overlap_with_training"]:
        setup_env.log_deviation(
            "cross-tier image duplication: {:.0f}% of test images and {:.0f}% of "
            "val images are byte-identical to a training-tier image".format(
                test_pct, val_pct),
            "tiers 0/1 are pooled entirely into training with no held-out split of "
            "their own (DENTEX paper); this is a property of the original design "
            "and the public release, inherited by any faithful reproduction",
            "01_setup_and_data",
            impact="the diagnosis-tier test/val split may not be held out from "
                   "everything the hierarchical pipeline has seen, at some tier, "
                   "during training")

tables.write_table(
    "image_overlap",
    [{"metric": key, "value": value} for key, value in overlap.items()
     if key != "per_split_unique_files"]
    + [{"metric": "unique_files__{}".format(name), "value": count}
       for name, count in overlap["per_split_unique_files"].items()],
    ["metric", "value"],
    "Cross-tier / cross-split image duplication by exact content hash (MD5). "
    "*_overlap_with_training counts diagnosis-tier test/val images that are "
    "byte-identical to an image somewhere in the pooled training tiers.",
    "01_setup_and_data", run.mode, "table:image_overlap")


In [ ]:
# ---- Notebook summary (the only cross-notebook contract) ----
summary = {
    "run_mode": run.mode,
    "environment": environment,
    "requirements_lock": lock,
    "vendored_modules": found,
    "download": {k: v for k, v in download.items() if k != "raw_train_json"},
    "paths": paths,
    "audits": audits,
    "image_overlap": overlap,
    "quadrant_box_geometry": quadrant_geometry,
    "clean_stress_confound": confound,
    "published_counts": actual_counts,
    "test_parse_report": {k: v for k, v in report.items() if k != "raw_label_counts"},
    "registration": registration_report,
    "subsets": {"path": subset_path, "rule": subsets["rule"],
                "clean": len(subsets["clean"]["image_ids"]),
                "stress": len(subsets["stress"]["image_ids"])},
    "dataset_hashes": hashes,
    "license_note": data_convert.LICENSE_NOTE,
}

path = setup_env.write_notebook_summary("01_setup_and_data", summary)
print("wrote", path)
print(json.dumps(summary, indent=2, default=str)[:4000])


In [ ]:
# ---- Publish, last: the summary above must be inside what gets published ----
# paper_assets/ ships with the data because it holds this notebook's summary
# (dataset hashes, audit counts), which notebook 03's reproducibility checklist
# reads back — and a fresh Kaggle session re-clones paper_assets/ empty.
publish = {"status": "disabled"}
if PUBLISH_KAGGLE_DATASET:
    publish = train_utils.publish_kaggle_dataset(
        DATA_DATASET_SLUG, [paths["root"], setup_env.PAPER_ASSETS],
        "converted DENTEX (run mode {})".format(run.mode))
    summary["kaggle_publish"] = {k: v for k, v in publish.items()
                                 if k not in ("stdout", "stderr")}
    setup_env.write_notebook_summary("01_setup_and_data", summary)
print(json.dumps({k: v for k, v in publish.items() if k not in ("stdout", "stderr")},
                 indent=2))
print("\nAttach this dataset to notebook 02 as: {}".format(DATA_DATASET_SLUG))
